# Abaqus Crack-Tip Boundary Conditions – Interactive App

This app evaluates **displacement boundary conditions for an Abaqus FE sub-model** from the
analytical crack-tip fields (isotropic, anisotropic single crystal, interface crack between
two anisotropic grains) and exports them as an Abaqus `*BOUNDARY` definition.

**Workflow:**
1. Load an Abaqus `.inp` file (nodes and node sets are read)
2. Choose model, material(s) and orientation(s)
3. Enter the stress intensity factors $K_I, K_{II}, K_{III}$, the crack length and the
   position of the crack tip in the global coordinate system
4. Select the node set(s) to which the boundary condition is applied
5. Compute & export → writes an `.inp` file with a `*BOUNDARY` block that can be included in
   your main model with `*INCLUDE, INPUT=<file>`

The physics (material database, Stroh formalism, coordinate transformation, `.inp` parser)
lives in the module **`crack_tip_fields.py`** in `../src` – see there for details and
the notebook `crack_tip_fields_validation_and_examples.ipynb` in this folder for derivation, examples and
validation.

> **Note on the .inp file:** The parser reads `*NODE`, `*ELEMENT` and `*NSET` blocks of a job
> input file written by Abaqus/CAE (part/instance or orphan mesh). The instance name is detected
> and used as node prefix (`Part-1-1.<node>`) in the exported boundary condition; the elements are
> needed to assign the duplicate crack-face nodes of a seam to the upper/lower crack face.
> Example models: `../examples/fe_case_study/2D_mode_I/<case>/*.inp` and, for 3D, `../examples/fe_case_study/3D_slice/slice_3D.inp`.

## 0. Setup

This single cell creates **all** widgets and callback functions of the app. Please always
run this cell first (or let "Run All" run through once) before using the buttons/dropdowns
further down – this ensures that all referenced widgets exist when you click, regardless of
which display cell below you execute.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))  # crack_tip_fields.py in ../src

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from crack_tip_fields import (
    MATERIALS, IsotropicCrackModel, AnisotropicCrackModel, InterfaceCrackModel,
    global_to_local, local_to_global_vector, parse_abaqus_mesh, abaqus_instance_name,
    crack_face_signs, export_abaqus_boundary,
    displacement_field_isoElast, stress_field_isoElast, compute_field_grid, plot_heatmap,
)

material_options = list(MATERIALS.keys())
state = {"nodes": {}, "node_sets": {}, "mesh": None}

# ---------------------------------------------------------------------------
# 1. Load file
# ---------------------------------------------------------------------------
path_w = widgets.Text(
    value="", placeholder="/path/to/file.inp", description="Path to .inp:",
    style={"description_width": "140px"}, layout=widgets.Layout(width="520px"),
)
load_btn = widgets.Button(description="Load file", button_style="primary", icon="upload")
load_out = widgets.Output()

# ---------------------------------------------------------------------------
# 2. Model, material and orientation
# ---------------------------------------------------------------------------
model_w = widgets.Dropdown(
    options=[("Isotropic", "isotropic"),
             ("Anisotropic (homogeneous single crystal)", "anisotropic"),
             ("Interface crack (two grains)", "interface")],
    value="isotropic", description="Model:", style={"description_width": "140px"}, layout=widgets.Layout(width="480px"),
)

material1_w = widgets.Dropdown(options=material_options, description="Material grain 1:",
                                style={"description_width": "140px"}, layout=widgets.Layout(width="480px"))
phi1_1_w = widgets.FloatText(value=0.0, description="phi1 [deg]:", layout=widgets.Layout(width="160px"))
Phi_1_w = widgets.FloatText(value=0.0, description="Phi [deg]:", layout=widgets.Layout(width="160px"))
phi2_1_w = widgets.FloatText(value=0.0, description="phi2 [deg]:", layout=widgets.Layout(width="160px"))
orient1_box = widgets.HBox([phi1_1_w, Phi_1_w, phi2_1_w])
grain1_box = widgets.VBox([material1_w, orient1_box])

material2_w = widgets.Dropdown(options=material_options, description="Material grain 2:",
                                style={"description_width": "140px"}, layout=widgets.Layout(width="480px"))
phi1_2_w = widgets.FloatText(value=0.0, description="phi1 [deg]:", layout=widgets.Layout(width="160px"))
Phi_2_w = widgets.FloatText(value=0.0, description="Phi [deg]:", layout=widgets.Layout(width="160px"))
phi2_2_w = widgets.FloatText(value=0.0, description="phi2 [deg]:", layout=widgets.Layout(width="160px"))
orient2_box = widgets.HBox([phi1_2_w, Phi_2_w, phi2_2_w])
grain2_label = widgets.HTML("<b>Grain 2 (below the interface)</b>")
grain2_box = widgets.VBox([grain2_label, material2_w, orient2_box])

# ---------------------------------------------------------------------------
# 3. Boundary conditions (K) and crack geometry
# ---------------------------------------------------------------------------
KI_w = widgets.FloatText(value=158.114, description="K_I [MPa*sqrt(mm)]:", style={"description_width": "160px"})
KII_w = widgets.FloatText(value=0.0, description="K_II [MPa*sqrt(mm)]:", style={"description_width": "160px"})
KIII_w = widgets.FloatText(value=0.0, description="K_III [MPa*sqrt(mm)]:", style={"description_width": "160px"})
a_w = widgets.FloatText(value=0.05, description="Crack half-length a [mm]:", style={"description_width": "160px"})
X0_w = widgets.FloatText(value=0.0, description="Crack tip X0 [mm]:", style={"description_width": "160px"})
Y0_w = widgets.FloatText(value=0.0, description="Crack tip Y0 [mm]:", style={"description_width": "160px"})
alpha_w = widgets.FloatText(value=0.0, description="Crack direction alpha [deg]:", style={"description_width": "160px"})

# ---------------------------------------------------------------------------
# 4. Node set(s)
# ---------------------------------------------------------------------------
nset_top_label = widgets.HTML("<b>Node set (all affected nodes)</b>")
nset_top_w = widgets.Dropdown(options=[], description="Node set:", style={"description_width": "140px"},
                              layout=widgets.Layout(width="480px"))
nset_bot_label = widgets.HTML("<b>Node set grain 2 (bottom)</b>")
nset_bot_w = widgets.Dropdown(options=[], description="Node set:", style={"description_width": "140px"},
                              layout=widgets.Layout(width="480px"))

# ---------------------------------------------------------------------------
# 5. Export options
# ---------------------------------------------------------------------------
dof1_w = widgets.Checkbox(value=True, description="Export u1 (DOF 1)")
dof2_w = widgets.Checkbox(value=True, description="Export u2 (DOF 2)")
dof3_w = widgets.Checkbox(value=False, description="Export u3 (DOF 3, e.g. generalised plane strain / 3D)")
instance_w = widgets.Text(value="", description="Instance prefix:",
                           placeholder="e.g. Part-1-1 (empty = plain node labels)",
                           style={"description_width": "140px"}, layout=widgets.Layout(width="400px"))
outfile_w = widgets.Text(value="bc_K.inp", description="Output file:",
                          style={"description_width": "140px"}, layout=widgets.Layout(width="400px"))
run_btn = widgets.Button(description="Compute & Export", button_style="success", icon="play")
run_out = widgets.Output()

# ---------------------------------------------------------------------------
# 6. Visualise crack-tip fields (with overlaid FE nodes)
# ---------------------------------------------------------------------------
show_disp_cb = widgets.Checkbox(value=True, description="Displacement fields (u1, u2, u3)")
show_stress_cb = widgets.Checkbox(value=True, description="Stress fields (sigma_11, sigma_22, sigma_12)")
grid_res_w = widgets.IntText(value=60, description="Grid resolution:", style={"description_width": "160px"})
sigma_cap_w = widgets.FloatText(value=2500., description="Stress cap [MPa]:", style={"description_width": "160px"})
zoom_radius_w = widgets.FloatText(value=0.0, description="Zoom radius [mm]:", style={"description_width": "160px"})
fields_btn = widgets.Button(description="Show crack-tip fields", button_style="info", icon="area-chart")
fields_out = widgets.Output()


# ---------------------------------------------------------------------------
# Callback functions
# ---------------------------------------------------------------------------
def update_model_visibility(change=None):
    kind = model_w.value
    orient1_box.layout.display = "none" if kind == "isotropic" else ""
    grain2_box.layout.display = "" if kind == "interface" else "none"
    nset_bot_w.layout.display = "" if kind == "interface" else "none"
    nset_bot_label.layout.display = "" if kind == "interface" else "none"
    nset_top_label.value = ("<b>Node set grain 1 (top)</b>" if kind == "interface"
                             else "<b>Node set (all affected nodes)</b>")


def on_load_clicked(b):
    with load_out:
        clear_output()
        path = path_w.value.strip()
        if not path or not os.path.isfile(path):
            print(f"File not found: {path!r}")
            return
        try:
            mesh = parse_abaqus_mesh(path)
        except Exception as e:
            print("Error while reading:", e)
            return
        nodes, node_sets = mesh["nodes"], mesh["node_sets"]
        state["mesh"] = mesh
        state["nodes"] = nodes
        state["node_sets"] = node_sets
        try:
            instance_w.value = abaqus_instance_name(mesh)
        except ValueError as e:
            instance_w.value = ""
            print("Note:", e)
        print(f"{len(mesh['elements'])} elements read (used to assign the crack faces).")
        print("Part instance(s): " + (", ".join(mesh["instances"]) if mesh["instances"]
                                     else "none (flat input file, plain node labels)"))
        set_names = sorted(node_sets.keys())
        nset_top_w.options = set_names
        nset_bot_w.options = set_names
        print(f"{len(nodes)} nodes read.")
        if set_names:
            print(f"{len(set_names)} node set(s) found:")
            for name in set_names:
                print(f"  - {name}: {len(node_sets[name])} nodes")
        else:
            print("No *NSET blocks found - please check the .inp file.")


def build_model():
    kind = model_w.value
    if kind == "isotropic":
        return IsotropicCrackModel(material1_w.value)
    elif kind == "anisotropic":
        return AnisotropicCrackModel(material1_w.value, phi1_1_w.value, Phi_1_w.value, phi2_1_w.value)
    else:
        return InterfaceCrackModel(
            material1_w.value, (phi1_1_w.value, Phi_1_w.value, phi2_1_w.value),
            material2_w.value, (phi1_2_w.value, Phi_2_w.value, phi2_2_w.value),
        )


def compute_displacement(model, kind, x1_l, x2_l, side):
    K = (KI_w.value, KII_w.value, KIII_w.value)
    if kind == "interface":
        return model.displacement(x1_l, x2_l, *K, a_w.value, side=side)
    else:
        return model.displacement(x1_l, x2_l, *K, a_w.value)


def on_run_clicked(b):
    with run_out:
        clear_output()
        if not state["nodes"]:
            print("Please load an .inp file first (section 1).")
            return
        kind = model_w.value
        try:
            model = build_model()
        except Exception as e:
            print("Error while building the model:", e)
            return

        if not nset_top_w.value:
            print("Please select a node set.")
            return
        node_ids = [(nid, "top") for nid in state["node_sets"][nset_top_w.value]]
        if kind == "interface":
            if not nset_bot_w.value:
                print("Please also select the node set for grain 2.")
                return
            if nset_bot_w.value == nset_top_w.value:
                print("The same node set is selected for grain 1 and grain 2 - please select the node set of "
                      "the lower grain for grain 2.")
                return
            node_ids += [(nid, "bot") for nid in state["node_sets"][nset_bot_w.value]]

        dof_list = tuple(d for d, cb in zip((1, 2, 3), (dof1_w, dof2_w, dof3_w)) if cb.value)
        if not dof_list:
            print("Please select at least one degree of freedom to export.")
            return

        # Crack faces: nodes on the crack line (x1 < 0, x2 = 0) exist twice (seam) with identical
        # coordinates; the face (upper/lower) follows from the attached elements and is passed to
        # the field functions as x2 = +0.0 / -0.0 (theta = +pi / -pi).
        face = {}
        mesh = state.get("mesh")
        if mesh is not None and mesh["elements"]:
            try:
                face = crack_face_signs([nid for nid, _ in node_ids], mesh["nodes"], mesh["elements"],
                                        (X0_w.value, Y0_w.value), alpha_w.value)
            except ValueError as e:
                print("Error while assigning the crack faces:", e)
                return
            if face:
                n_up = sum(1 for s in face.values() if s > 0)
                print(f"{len(face)} crack-face node(s) on the crack line: {n_up} upper, {len(face) - n_up} lower face.")
        else:
            print("Warning: no elements in the .inp file - crack-face nodes (x1 < 0, x2 = 0) cannot be "
                  "assigned to the upper/lower face and are evaluated on the upper face.")

        entries = []
        coords_for_plot = []
        umag_for_plot = []
        missing = []
        for nid, side in node_ids:
            if nid not in state["nodes"]:
                missing.append(nid)
                continue
            X, Y, Z = state["nodes"][nid]
            x1_l, x2_l = global_to_local(np.array(X), np.array(Y), (X0_w.value, Y0_w.value), alpha_w.value)
            x1_l, x2_l = float(x1_l), float(x2_l)
            if nid in face:
                x2_l = np.copysign(0.0, face[nid])
            try:
                u1, u2, u3 = compute_displacement(model, kind, x1_l, x2_l, side)
            except Exception as e:
                print(f"Error at node {nid}: {e}")
                return
            Ux, Uy = local_to_global_vector(u1, u2, alpha_w.value)
            values_by_dof = {1: Ux, 2: Uy, 3: u3}
            entries.append((nid, [values_by_dof[d] for d in dof_list]))
            coords_for_plot.append((X, Y))
            umag_for_plot.append(np.sqrt(Ux**2 + Uy**2 + u3**2))

        if missing:
            print(f"Warning: {len(missing)} node ID(s) of the node set not found among the loaded nodes: {missing[:10]}{' ...' if len(missing) > 10 else ''}")

        header = (f"Crack-tip field boundary condition ({kind})\n"
                  f"K_I={KI_w.value}, K_II={KII_w.value}, K_III={KIII_w.value} [MPa*sqrt(mm)]\n"
                  f"Crack tip=({X0_w.value},{Y0_w.value}) mm, alpha={alpha_w.value} deg, a={a_w.value} mm\n"
                  f"Node set(s): {nset_top_w.value}" + (f", {nset_bot_w.value}" if kind == "interface" else "")
                  + f"\nInstance prefix: {instance_w.value.strip() or '(none)'}")

        export_abaqus_boundary(outfile_w.value, entries, dof_list=dof_list, header_comment=header,
                               instance=instance_w.value.strip())

        print(f"{len(entries)} nodes exported to '{outfile_w.value}'.")
        print(f"Include in the Abaqus main model with:  *INCLUDE, INPUT={outfile_w.value}\n")
        preview_lines = open(outfile_w.value).read().splitlines()
        print("--- Preview (first 15 lines) ---")
        print("\n".join(preview_lines[:15]))
        if len(preview_lines) > 15:
            print(f"... ({len(preview_lines) - 15} more lines)")

        # Sanity plot: displacement magnitude at the exported nodes
        coords_for_plot = np.array(coords_for_plot)
        umag_for_plot = np.array(umag_for_plot)
        plt.figure(figsize=(5.5, 4.5))
        sc = plt.scatter(coords_for_plot[:, 0], coords_for_plot[:, 1], c=umag_for_plot, cmap="viridis", s=40)
        plt.colorbar(sc, label="|U| [mm]")
        plt.scatter([X0_w.value], [Y0_w.value], marker="x", color="red", s=80, label="Crack tip")
        plt.xlabel("X (global) [mm]"); plt.ylabel("Y (global) [mm]")
        plt.title("Exported boundary condition: displacement magnitude")
        plt.axis("equal"); plt.legend(); plt.tight_layout(); plt.show(); plt.close()


def _local_coords_of(node_ids):
    """Local (x1,x2) coordinates of a list of node IDs (invalid IDs are skipped)."""
    xs, ys = [], []
    for nid in node_ids:
        if nid not in state["nodes"]:
            continue
        X, Y, Z = state["nodes"][nid]
        x1_l, x2_l = global_to_local(np.array(X), np.array(Y), (X0_w.value, Y0_w.value), alpha_w.value)
        xs.append(float(x1_l)); ys.append(float(x2_l))
    return np.array(xs), np.array(ys)


def _auto_vlim(F, floor=1e-30):
    """Colour-scale limits (vmin, vmax) = actual value range of F - NOT forced symmetric
    about 0, so that purely positive (or purely negative) fields use the full colour range."""
    vmin, vmax = float(np.min(F)), float(np.max(F))
    if vmax - vmin < floor:
        vmax = vmin + floor
    return vmin, vmax


def _auto_vlim_pair(Fa, Fb, floor=1e-30):
    """Like _auto_vlim, but common to two fields (e.g. upper/lower grain), so that both
    sub-plots use the same, comparable colour scale."""
    vmin = min(float(np.min(Fa)), float(np.min(Fb)))
    vmax = max(float(np.max(Fa)), float(np.max(Fb)))
    if vmax - vmin < floor:
        vmax = vmin + floor
    return vmin, vmax


def _points_in_domain(px, py, x1_lo, x1_hi, x2_lo, x2_hi):
    """Filter points to the visible grid region (otherwise outlying points distort the axes)."""
    if len(px) == 0:
        return px, py
    mask = (px >= x1_lo) & (px <= x1_hi) & (py >= x2_lo) & (py <= x2_hi)
    return px[mask], py[mask]


def _render_isotropic_or_aniso(kind, model, K, x1_lo, x1_hi, x2_lo, x2_hi, n, sigma_cap, pts, suffix):
    x_vals = np.linspace(x1_lo, x1_hi, n)
    y_vals = np.linspace(x2_lo, x2_hi, n)
    X1, X2 = np.meshgrid(x_vals, y_vals)
    px, py = _points_in_domain(pts[0], pts[1], x1_lo, x1_hi, x2_lo, x2_hi)

    if kind == "isotropic":
        if show_disp_cb.value:
            U1, U2, U3 = displacement_field_isoElast(X1, X2, model.G, model.kappa, *K)
            for name, F in [("u1", U1), ("u2", U2), ("u3", U3)]:
                vmin, vmax = _auto_vlim(F)
                plot_heatmap(F, X1, X2, f"Displacement {name} (isotropic){suffix}", f"{name} [mm]",
                             vmin=vmin, vmax=vmax, points=(px, py))
        if show_stress_cb.value:
            sxx, syy, szz, sxy, sxz, syz = stress_field_isoElast(*K, X1, X2)
            for name, F in [("sigma_11", sxx), ("sigma_22", syy), ("sigma_12", sxy)]:
                Fc = np.clip(F, -sigma_cap, sigma_cap)
                vmin, vmax = _auto_vlim(Fc)
                plot_heatmap(Fc, X1, X2, f"{name} (isotropic){suffix}", f"{name} [MPa]",
                             vmin=vmin, vmax=vmax, points=(px, py))
    else:  # anisotropic
        def u_func(x1, x2):
            return model.displacement(x1, x2, *K, a_w.value)

        U1, U2, U3, SIG = compute_field_grid(u_func, model.C_rot, x_vals, y_vals, a_w.value,
                                             stress_func=lambda x, y: model.stress(x, y, *K, a_w.value))
        if show_disp_cb.value:
            for name, F in [("u1", U1), ("u2", U2), ("u3", U3)]:
                vmin, vmax = _auto_vlim(F)
                plot_heatmap(F, X1, X2, f"Displacement {name} (anisotropic, single crystal){suffix}", f"{name} [mm]",
                             vmin=vmin, vmax=vmax, points=(px, py))
        if show_stress_cb.value:
            for name, idx in [("sigma_11", 0), ("sigma_22", 1), ("sigma_12", 5)]:
                Fc = np.clip(SIG[:, :, idx], -sigma_cap, sigma_cap)
                vmin, vmax = _auto_vlim(Fc)
                plot_heatmap(Fc, X1, X2, f"{name} (anisotropic, single crystal){suffix}", f"{name} [MPa]",
                             vmin=vmin, vmax=vmax, points=(px, py))


def _render_interface(model, K, x1_lo, x1_hi, x2_extent, n, sigma_cap, pts_top, pts_bot, suffix):
    eps = 1e-9
    x_vals = np.linspace(x1_lo, x1_hi, n)
    y_top_vals = np.linspace(eps, x2_extent, max(5, n // 2))
    y_bot_vals = np.linspace(-x2_extent, -eps, max(5, n // 2))
    X1t, X2t = np.meshgrid(x_vals, y_top_vals)
    X1b, X2b = np.meshgrid(x_vals, y_bot_vals)

    px_top, py_top = _points_in_domain(pts_top[0], pts_top[1], x1_lo, x1_hi, 0.0, x2_extent)
    px_bot, py_bot = _points_in_domain(pts_bot[0], pts_bot[1], x1_lo, x1_hi, -x2_extent, 0.0)

    def u_func_top(x1, x2):
        return model.displacement(x1, x2, *K, a_w.value, side="top")

    def u_func_bot(x1, x2):
        return model.displacement(x1, x2, *K, a_w.value, side="bot")

    U1t, U2t, U3t, SIGt = compute_field_grid(u_func_top, model.C_rot_top, x_vals, y_top_vals, a_w.value,
                                          stress_func=lambda x, y: model.stress(x, y, *K, a_w.value, side="top"))
    U1b, U2b, U3b, SIGb = compute_field_grid(u_func_bot, model.C_rot_bot, x_vals, y_bot_vals, a_w.value,
                                          stress_func=lambda x, y: model.stress(x, y, *K, a_w.value, side="bot"))

    if show_disp_cb.value:
        for name, Ft, Fb in [("u1", U1t, U1b), ("u2", U2t, U2b), ("u3", U3t, U3b)]:
            vmin, vmax = _auto_vlim_pair(Ft, Fb)
            fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
            plot_heatmap(Ft, X1t, X2t, f"{name} (grain 1, top){suffix}", f"{name} [mm]",
                         vmin=vmin, vmax=vmax, points=(px_top, py_top), ax=axes[0])
            plot_heatmap(Fb, X1b, X2b, f"{name} (grain 2, bottom){suffix}", f"{name} [mm]",
                         vmin=vmin, vmax=vmax, points=(px_bot, py_bot), ax=axes[1])
            plt.tight_layout(); plt.show(); plt.close(fig)
    if show_stress_cb.value:
        for name, idx in [("sigma_11", 0), ("sigma_22", 1), ("sigma_12", 5)]:
            Ft = np.clip(SIGt[:, :, idx], -sigma_cap, sigma_cap)
            Fb = np.clip(SIGb[:, :, idx], -sigma_cap, sigma_cap)
            vmin, vmax = _auto_vlim_pair(Ft, Fb)
            fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
            plot_heatmap(Ft, X1t, X2t, f"{name} (grain 1, top){suffix}", f"{name} [MPa]",
                         vmin=vmin, vmax=vmax, points=(px_top, py_top), ax=axes[0])
            plot_heatmap(Fb, X1b, X2b, f"{name} (grain 2, bottom){suffix}", f"{name} [MPa]",
                         vmin=vmin, vmax=vmax, points=(px_bot, py_bot), ax=axes[1])
            plt.tight_layout(); plt.show(); plt.close(fig)


def on_show_fields_clicked(b):
    with fields_out:
        clear_output()
        if not state["nodes"]:
            print("Please load an .inp file first (section 1).")
            return
        kind = model_w.value
        try:
            model = build_model()
        except Exception as e:
            print("Error while building the model:", e)
            return
        if not nset_top_w.value:
            print("Please select a node set (section 4).")
            return

        x1_top, x2_top = _local_coords_of(state["node_sets"][nset_top_w.value])
        if kind == "interface":
            if not nset_bot_w.value:
                print("Please also select the node set for grain 2 (section 4).")
                return
            x1_bot, x2_bot = _local_coords_of(state["node_sets"][nset_bot_w.value])
            all_x1 = np.concatenate([x1_top, x1_bot]) if len(x1_bot) else x1_top
            all_x2 = np.concatenate([x2_top, x2_bot]) if len(x2_bot) else x2_top
        else:
            all_x1, all_x2 = x1_top, x2_top

        if len(all_x1) == 0:
            print("No valid nodes found in the selected node set.")
            return

        n = max(10, grid_res_w.value)
        K = (KI_w.value, KII_w.value, KIII_w.value)
        sigma_cap = sigma_cap_w.value
        pad = 0.2
        min_dist = float(np.sqrt(all_x1**2 + all_x2**2).min())

        # --- View 1: grid region around the nodes that are actually evaluated ---
        print("=== View 1: region of the evaluated FE nodes ===")
        x1_span = (all_x1.max() - all_x1.min()) or a_w.value
        x2_span = (all_x2.max() - all_x2.min()) or a_w.value
        x1_lo, x1_hi = all_x1.min() - pad * x1_span, all_x1.max() + pad * x1_span

        if kind == "interface":
            x2_extent_nodes = max(np.abs(all_x2).max(), a_w.value * 0.05) * (1 + pad)
            _render_interface(model, K, x1_lo, x1_hi, x2_extent_nodes, n, sigma_cap,
                               (x1_top, x2_top), (x1_bot, x2_bot), suffix="")
        else:
            x2_lo, x2_hi = all_x2.min() - pad * x2_span, all_x2.max() + pad * x2_span
            _render_isotropic_or_aniso(kind, model, K, x1_lo, x1_hi, x2_lo, x2_hi, n, sigma_cap,
                                        (all_x1, all_x2), suffix="")

        # --- View 2: zoom on the crack tip, independent of the node coordinates ---
        zoom_r = zoom_radius_w.value if zoom_radius_w.value > 0 else 0.1 * a_w.value
        print(f"\n=== View 2: zoom on the crack tip (radius = {zoom_r:g} mm) ===")
        if min_dist > zoom_r:
            print(f"Note: The nearest evaluated node is {min_dist:.4g} mm from "
                  f"the crack tip - outside this zoom region. The singularity "
                  f"is shown here for orientation, but no FE node lies within the plot area. "
                  f"If this is surprising: check the crack tip (X0,Y0) and a in section 3.")

        if kind == "interface":
            _render_interface(model, K, -zoom_r, zoom_r, zoom_r, n, sigma_cap,
                               (x1_top, x2_top), (x1_bot, x2_bot), suffix=" - crack-tip zoom")
        else:
            _render_isotropic_or_aniso(kind, model, K, -zoom_r, zoom_r, -zoom_r, zoom_r, n, sigma_cap,
                                        (all_x1, all_x2), suffix=" - crack-tip zoom")

        print("\nDone. Circles = the FE nodes actually evaluated for the export "
              "(only visible if they lie within the respective plot area). Colour scales are "
              "scaled automatically to the actual value range of each plot.")


# ---------------------------------------------------------------------------
# Wire up the widgets (attach callbacks) + set the initial visibility state
# ---------------------------------------------------------------------------
load_btn.on_click(on_load_clicked)
model_w.observe(update_model_visibility, names="value")
run_btn.on_click(on_run_clicked)
fields_btn.on_click(on_show_fields_clicked)
update_model_visibility()

print("Setup complete - widgets are ready. Continue with section 1 below.")


Setup complete - widgets are ready. Continue with section 1 below.


## 1. Load Abaqus file

In [2]:
display(widgets.VBox([widgets.HBox([path_w, load_btn]), load_out]))


## 2. Model, material and orientation

In [3]:
display(widgets.VBox([model_w, grain1_box, grain2_box]))


## 3. Boundary conditions ($K_I, K_{II}, K_{III}$) and crack geometry

In [4]:
display(widgets.VBox([
    widgets.HTML("<b>Stress intensity factors</b> (unit: MPa*sqrt(mm); MPa*sqrt(m) / sqrt(1000))"),
    widgets.HBox([KI_w, KII_w, KIII_w]),
    widgets.HTML("<b>Crack geometry in the global (Abaqus) coordinate system</b><br>"
                 "alpha: angle between the global X axis and the local crack-propagation direction x1"),
    widgets.HBox([a_w, X0_w, Y0_w, alpha_w]),
]))

## 4. Select node set(s)

In [5]:
display(widgets.VBox([nset_top_label, nset_top_w, nset_bot_label, nset_bot_w]))


## 5. Export options and execution

In [6]:
display(widgets.VBox([
    widgets.HBox([dof1_w, dof2_w, dof3_w]),
    instance_w,
    widgets.HBox([outfile_w, run_btn]),
    run_out,
]))


## 6. Visualise the crack-tip fields

Shows the displacement/stress fields as heatmaps, as in the validation notebook – **in two
views**:

1. **Node region**: grid region around the FE nodes that are actually evaluated (black
   circles), scaled automatically.
2. **Zoom on the crack tip**: a small region around the crack tip that is independent of the
   node coordinates (adjustable radius, default = 10 % of the crack half-length `a`), so that
   the $1/\sqrt r$ singularity itself is well resolved – even if the FE nodes (e.g. of a large
   sub-model) are far from the crack tip, where almost nothing of the singularity is visible
   in the field.

The colour scale is scaled automatically to the actual value range of each plot (no fixed
clipping that makes small values look "all zero") – the stress cap only limits extreme values
very close to the singularity.

> For the anisotropic and interface models the stresses are evaluated in closed form
> (`crack_tip_fields.compute_field_grid` with `stress_func`); the Stroh formalism is not
> vectorised, so fine grids (and the two views) can still take a few seconds.

In [7]:
display(widgets.VBox([
    widgets.HBox([show_disp_cb, show_stress_cb]),
    widgets.HBox([grid_res_w, sigma_cap_w, zoom_radius_w]),
    widgets.HTML("<i>Zoom radius = 0 -> automatically 10% of the crack half-length a</i>"),
    fields_btn,
    fields_out,
]))